In [18]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import yfinance as yf
from arch import arch_model
import os

In [19]:
TRADING_DAYS = 245
HORIZON = 10

# All trading days from 2025-12-01 through today
CALC_DATES = pd.bdate_range(start='2025-12-01', end=pd.Timestamp.today().normalize())

# Keep scalar CALC_DATE for existing downstream code
CALC_DATE = CALC_DATES.max()

#PRICE_START_DATE = CALC_DATE - pd.tseries.offsets.BDay(TRADING_DAYS)
#PRICE_END_DATE = CALC_DATE
STRESS_DATE_START = pd.to_datetime('2008-07-01')
STRESS_DATE_END = pd.to_datetime('2009-03-31')

SIM_DAYS = 10
N_SIMS = 10_000

In [20]:
for date in CALC_DATES:
    PRICE_START_DATE = date - pd.tseries.offsets.BDay(TRADING_DAYS)
    PRICE_END_DATE = date
    #setup output directory
    date_str = date.strftime('%Y%m%d')
    output_dir = f'outputs/{date_str}'

    os.makedirs(output_dir, exist_ok=True)



    #load portfolio and get positions
    portfolio_df = pd.read_csv("portfolio.csv")
    portfolio_df['ticker'] = portfolio_df['ticker'].astype(str)
    tickers = portfolio_df['ticker'].unique().tolist()

    positions_df = portfolio_df.copy()

    positions_df['purchase_date'] = pd.to_datetime(positions_df['purchase_date'])
    positions_df = positions_df[positions_df["purchase_date"] <= date]

    positions_df['signed_qty'] = positions_df['quantity']
    positions_df.loc[positions_df['buy_sell'] == 'sell', 'signed_qty'] *= -1

    positions = positions_df.groupby("ticker")["signed_qty"].sum()
    positions = positions[positions != 0]



    # get price data
    price_data = yf.download(
        tickers=positions.index.tolist(),
        start=PRICE_START_DATE,
        end=PRICE_END_DATE,
        group_by='ticker',
        auto_adjust=True,
        progress=False
    )

    price_data = price_data.loc[:, (slice(None), ['Close', 'Volume'])]
    close_prices = price_data.xs('Close', level=1, axis=1)
    log_returns = np.log(close_prices / close_prices.shift(1)).dropna()

    stress_data = yf.download(
        tickers=tickers,
        start=STRESS_DATE_START,
        end=STRESS_DATE_END,
        group_by='ticker',
        auto_adjust=True,
        progress=False
    )

    stress_data = stress_data.loc[:, (slice(None), ['Close', 'Volume'])]
    stress_close_prices = stress_data.xs('Close', level=1, axis=1)
    stress_log_returns = np.log(stress_close_prices / stress_close_prices.shift(1)).dropna()



    def get_pnls(positions, close_prices, log_returns):
        # get params of each asset
        garch_params = {}
        manual_forecasts = {}
        tickers = log_returns.columns
        
        for ticker in log_returns.columns:
            r = log_returns[ticker] * 100  # convert to %
        
            model = arch_model(
                r,
                vol='Garch',
                p=1,
                q=1,
                mean='Constant',
                dist='normal'  # can change to 't' later
            )
        
            res = model.fit(disp='off')
        
            params = res.params
            garch_params[ticker] = {
                'mu': params['mu'] / 100,
                'omega': params['omega'],
                'alpha': params['alpha[1]'],
                'beta': params['beta[1]'],
                'sigma': res.conditional_volatility.iloc[-1] / 100
            }

        # idk
        n_assets = len(tickers)

        corr = log_returns.corr().values
        L = np.linalg.cholesky(corr)

        # get returns
        sim_returns = np.zeros((N_SIMS, SIM_DAYS, n_assets))

        for sim in range(N_SIMS):
        
            # initialise vol per asset
            sigma_t = np.array([garch_params[t]['sigma'] for t in tickers])
            sigma2_t = sigma_t**2
        
            for t in range(SIM_DAYS):
        
                # correlated shocks
                z = np.random.normal(size=n_assets)
                z_corr = L @ z
        
                epsilon = sigma_t * z_corr
        
                # store returns
                mu_vec = np.array([garch_params[t]['mu'] for t in tickers])
                sim_returns[sim, t, :] = mu_vec + epsilon
        
                # update variance
                for i, ticker in enumerate(tickers):
                    p = garch_params[ticker]
                    sigma2_t[i] = (
                        p['omega']
                        + p['alpha'] * (epsilon[i] * 100)**2
                        + p['beta'] * sigma2_t[i]
                    )
        
                sigma_t = np.sqrt(sigma2_t) / 100

        # get final pnls
        S0 = close_prices.iloc[-1].values.reshape(1, -1)
        cum_log_returns = sim_returns.sum(axis=1)  # shape: (n_sims, n_assets)
        final_prices = S0 * np.exp(cum_log_returns)
        final_prices.shape = (N_SIMS, n_assets)
        positions_vec = positions.reindex(close_prices.columns).fillna(0).values
        asset_pnl = (final_prices - S0) * positions_vec
        pnl = asset_pnl.sum(axis=1)

        # get risk metrics
        VaR_95 = np.percentile(pnl, 5)
        VaR_99 = np.percentile(pnl, 1)
        ES_95 = pnl[pnl <= VaR_95].mean()
        ES_99 = pnl[pnl <= VaR_99].mean()

        # return
        #return [pnl, VaR_95, VaR_99, ES_95]
        return {
            "pnls": pnl,
            "VaR_95": VaR_95,
            "VaR_99": VaR_99,
            "ES_95": ES_95,
            "ES_99": ES_99
        }




    # get base risk and pnls
    all_pnls = get_pnls(positions, close_prices, log_returns)
    date_str = date.strftime('%Y%m%d')

    pnls_df = pd.DataFrame({'pnl': all_pnls['pnls']})
    #pnls_df.to_csv(f'outputs/{date_str}/pnls.csv', index=False)

    risk_df = pd.DataFrame([
        {
            'Portfolio Value': (positions * close_prices.iloc[-1]).sum(),
            'VaR_95': all_pnls['VaR_95'],
            'VaR_99': all_pnls['VaR_99'],
            'ES_95': all_pnls['ES_95'],
            'ES_99': all_pnls['ES_99']
        }
    ])

    pnls_df.to_csv(f'{output_dir}/pnls.csv', index=False)
    risk_df.to_csv(f'{output_dir}/risk.csv', index=False)

    # marginal risk
    marginal_risk_rows = []

    for ticker in positions.index:
        positions_excluded = positions.copy()
        positions_excluded.loc[ticker] = 0
        excluded_results = get_pnls(positions_excluded, close_prices, log_returns)

        marginal_risk_rows.append({
            'excluded_ticker': ticker,
            'VaR_95': excluded_results['VaR_95'],
            'VaR_99': excluded_results['VaR_99'],
            'ES_95': excluded_results['ES_95'],
            'ES_99': excluded_results['ES_99']
        })

    marginal_risk_df = pd.DataFrame(marginal_risk_rows)
    marginal_risk_df.to_csv(f'{output_dir}/marginal_risk.csv', index=False)

    #stress risk
    stress_pnls = get_pnls(positions, close_prices, stress_log_returns)
    stress_df = pd.DataFrame([
        {
            'VaR_95': stress_pnls['VaR_95'],
            'VaR_99': stress_pnls['VaR_99'],
            'ES_95': stress_pnls['ES_95'],
            'ES_99': stress_pnls['ES_99']
        }
    ])
    stress_df.to_csv(f'{output_dir}/stress_risk.csv', index=False)
    print("Completed calculations for date:", date_str)

Completed calculations for date: 20251201
Completed calculations for date: 20251202
Completed calculations for date: 20251203
Completed calculations for date: 20251204
Completed calculations for date: 20251205
Completed calculations for date: 20251208
Completed calculations for date: 20251209
Completed calculations for date: 20251210
Completed calculations for date: 20251211
Completed calculations for date: 20251212
Completed calculations for date: 20251215
Completed calculations for date: 20251216
Completed calculations for date: 20251217
Completed calculations for date: 20251218
Completed calculations for date: 20251219
Completed calculations for date: 20251222
Completed calculations for date: 20251223
Completed calculations for date: 20251224
Completed calculations for date: 20251225
Completed calculations for date: 20251226
Completed calculations for date: 20251229
Completed calculations for date: 20251230
Completed calculations for date: 20251231
Completed calculations for date: 2

In [21]:
'''#setup output directory
date_str = CALC_DATE.strftime('%Y%m%d')
output_dir = f'outputs/{date_str}'

os.makedirs(output_dir, exist_ok=True)



#load portfolio and get positions
portfolio_df = pd.read_csv("portfolio.csv")
portfolio_df['ticker'] = portfolio_df['ticker'].astype(str)
tickers = portfolio_df['ticker'].unique().tolist()

positions_df = portfolio_df.copy()

positions_df['purchase_date'] = pd.to_datetime(positions_df['purchase_date'])
positions_df = positions_df[positions_df["purchase_date"] <= CALC_DATE]

positions_df['signed_qty'] = positions_df['quantity']
positions_df.loc[positions_df['buy_sell'] == 'sell', 'signed_qty'] *= -1

positions = positions_df.groupby("ticker")["signed_qty"].sum()
positions = positions[positions != 0]



# get price data
price_data = yf.download(
    tickers=positions.index.tolist(),
    start=PRICE_START_DATE,
    end=PRICE_END_DATE,
    group_by='ticker',
    auto_adjust=True,
    progress=False
)

price_data = price_data.loc[:, (slice(None), ['Close', 'Volume'])]
close_prices = price_data.xs('Close', level=1, axis=1)
log_returns = np.log(close_prices / close_prices.shift(1)).dropna()

stress_data = yf.download(
    tickers=tickers,
    start=STRESS_DATE_START,
    end=STRESS_DATE_END,
    group_by='ticker',
    auto_adjust=True,
    progress=False
)

stress_data = stress_data.loc[:, (slice(None), ['Close', 'Volume'])]
stress_close_prices = stress_data.xs('Close', level=1, axis=1)
stress_log_returns = np.log(stress_close_prices / stress_close_prices.shift(1)).dropna()



def get_pnls(positions, close_prices, log_returns):
    # get params of each asset
    garch_params = {}
    manual_forecasts = {}
    tickers = log_returns.columns
    
    for ticker in log_returns.columns:
        r = log_returns[ticker] * 100  # convert to %
    
        model = arch_model(
            r,
            vol='Garch',
            p=1,
            q=1,
            mean='Constant',
            dist='normal'  # can change to 't' later
        )
    
        res = model.fit(disp='off')
    
        params = res.params
        garch_params[ticker] = {
            'mu': params['mu'] / 100,
            'omega': params['omega'],
            'alpha': params['alpha[1]'],
            'beta': params['beta[1]'],
            'sigma': res.conditional_volatility.iloc[-1] / 100
        }

    # idk
    n_assets = len(tickers)

    corr = log_returns.corr().values
    L = np.linalg.cholesky(corr)

    # get returns
    sim_returns = np.zeros((N_SIMS, SIM_DAYS, n_assets))

    for sim in range(N_SIMS):
    
        # initialise vol per asset
        sigma_t = np.array([garch_params[t]['sigma'] for t in tickers])
        sigma2_t = sigma_t**2
    
        for t in range(SIM_DAYS):
    
            # correlated shocks
            z = np.random.normal(size=n_assets)
            z_corr = L @ z
    
            epsilon = sigma_t * z_corr
    
            # store returns
            mu_vec = np.array([garch_params[t]['mu'] for t in tickers])
            sim_returns[sim, t, :] = mu_vec + epsilon
    
            # update variance
            for i, ticker in enumerate(tickers):
                p = garch_params[ticker]
                sigma2_t[i] = (
                    p['omega']
                    + p['alpha'] * (epsilon[i] * 100)**2
                    + p['beta'] * sigma2_t[i]
                )
    
            sigma_t = np.sqrt(sigma2_t) / 100

    # get final pnls
    S0 = close_prices.iloc[-1].values.reshape(1, -1)
    cum_log_returns = sim_returns.sum(axis=1)  # shape: (n_sims, n_assets)
    final_prices = S0 * np.exp(cum_log_returns)
    final_prices.shape = (N_SIMS, n_assets)
    positions_vec = positions.reindex(close_prices.columns).fillna(0).values
    asset_pnl = (final_prices - S0) * positions_vec
    pnl = asset_pnl.sum(axis=1)

    # get risk metrics
    VaR_95 = np.percentile(pnl, 5)
    VaR_99 = np.percentile(pnl, 1)
    ES_95 = pnl[pnl <= VaR_95].mean()
    ES_99 = pnl[pnl <= VaR_99].mean()

    # return
    #return [pnl, VaR_95, VaR_99, ES_95]
    return {
        "pnls": pnl,
        "VaR_95": VaR_95,
        "VaR_99": VaR_99,
        "ES_95": ES_95,
        "ES_99": ES_99
    }




# get base risk and pnls
all_pnls = get_pnls(positions, close_prices, log_returns)
date_str = CALC_DATE.strftime('%Y%m%d')

pnls_df = pd.DataFrame({'pnl': all_pnls['pnls']})
#pnls_df.to_csv(f'outputs/{date_str}/pnls.csv', index=False)

risk_df = pd.DataFrame([
    {
        'VaR_95': all_pnls['VaR_95'],
        'VaR_99': all_pnls['VaR_99'],
        'ES_95': all_pnls['ES_95'],
        'ES_99': all_pnls['ES_99']
    }
])

pnls_df.to_csv(f'{output_dir}/pnls.csv', index=False)
risk_df.to_csv(f'{output_dir}/risk.csv', index=False)

# marginal risk
marginal_risk_rows = []

for ticker in positions.index:
    positions_excluded = positions.copy()
    positions_excluded.loc[ticker] = 0
    excluded_results = get_pnls(positions_excluded, close_prices, log_returns)

    marginal_risk_rows.append({
        'excluded_ticker': ticker,
        'VaR_95': excluded_results['VaR_95'],
        'VaR_99': excluded_results['VaR_99'],
        'ES_95': excluded_results['ES_95'],
        'ES_99': excluded_results['ES_99']
    })

marginal_risk_df = pd.DataFrame(marginal_risk_rows)
marginal_risk_df.to_csv(f'{output_dir}/marginal_risk.csv', index=False)

#stress risk
stress_pnls = get_pnls(positions, close_prices, stress_log_returns)
stress_df = pd.DataFrame([
    {
        'VaR_95': stress_pnls['VaR_95'],
        'VaR_99': stress_pnls['VaR_99'],
        'ES_95': stress_pnls['ES_95'],
        'ES_99': stress_pnls['ES_99']
    }
])
stress_df.to_csv(f'{output_dir}/stress_risk.csv', index=False)'''

'#setup output directory\ndate_str = CALC_DATE.strftime(\'%Y%m%d\')\noutput_dir = f\'outputs/{date_str}\'\n\nos.makedirs(output_dir, exist_ok=True)\n\n\n\n#load portfolio and get positions\nportfolio_df = pd.read_csv("portfolio.csv")\nportfolio_df[\'ticker\'] = portfolio_df[\'ticker\'].astype(str)\ntickers = portfolio_df[\'ticker\'].unique().tolist()\n\npositions_df = portfolio_df.copy()\n\npositions_df[\'purchase_date\'] = pd.to_datetime(positions_df[\'purchase_date\'])\npositions_df = positions_df[positions_df["purchase_date"] <= CALC_DATE]\n\npositions_df[\'signed_qty\'] = positions_df[\'quantity\']\npositions_df.loc[positions_df[\'buy_sell\'] == \'sell\', \'signed_qty\'] *= -1\n\npositions = positions_df.groupby("ticker")["signed_qty"].sum()\npositions = positions[positions != 0]\n\n\n\n# get price data\nprice_data = yf.download(\n    tickers=positions.index.tolist(),\n    start=PRICE_START_DATE,\n    end=PRICE_END_DATE,\n    group_by=\'ticker\',\n    auto_adjust=True,\n    progres